In [4]:
import pandas as pd
from pathlib import Path

In [14]:
DATA_PATH = Path("../data/raw")

train_df = pd.read_csv(DATA_PATH / "cs-training.csv")
# Preview the data
print(train_df.shape)

(150000, 12)


In [18]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

# Columns
* SeriousDlqin2yrs (target)
* RevolvingUtilizationOfUnsecuredLines
* age
* NumberOfTime30-59DaysPastDueNotWorse
* DebtRatio
* MonthlyIncome
* NumberOfOpenCreditLinesAndLoans
* NumberOfTimes90DaysLate
* NumberRealEstateLoansOrLines
* NumberOfTime60-89DaysPastDueNotWorse
* NumberOfDependents (has nulls)

## Number of dependents treatment

In [22]:
missing_pct = train_df['NumberOfDependents'].isna().mean() * 100
print(f"{missing_pct:.2f}% missing")

2.62% missing


In [26]:
train_df['NumberOfDependents'].describe()

count    146076.000000
mean          0.757222
std           1.115086
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          20.000000
Name: NumberOfDependents, dtype: float64

In [ ]:
# Median is 0, we will fill NaNs with 0
train_df['NumberOfDependents'] = (
    train_df['NumberOfDependents']
    .fillna(train_df['NumberOfDependents'].median())
)
# Given that its integer, we will swap from float to integer
train_df['NumberOfDependents'] = (
    train_df['NumberOfDependents']
    .fillna(train_df['NumberOfDependents'].median())
    .astype(int)
)

In [29]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

# Monthly income treatment

In [34]:
print("Percentage missing: ", train_df['MonthlyIncome'].isna().mean() * 100)

Percentage missing:  19.820666666666668


In [ ]:
train_df['MonthlyIncome'].describe()

train_df['MonthlyIncome'].isna().sum()

count    1.500000e+05
mean     6.418455e+03
std      1.289040e+04
min      0.000000e+00
25%      3.903000e+03
50%      5.400000e+03
75%      7.400000e+03
max      3.008750e+06
Name: MonthlyIncome, dtype: float64

In [ ]:
# Given that income missing is almost 20% missing and imputting information would be dangerous
# We fill it with the median
# We add a new column to show that it had to be filled
train_df['IncomeMissing'] = (
    train_df['MonthlyIncome']
    .isna()
    .astype(int)
)

median_income = train_df['MonthlyIncome'].median()

train_df['MonthlyIncome'] = (
    train_df['MonthlyIncome']
    .fillna(median_income)
)

MonthlyIncome contained missing values for 29,731 applicants (19.8% of the dataset). Missing values were imputed using the median, and a binary IncomeMissing indicator was created. Applicants with missing income had a default rate of 5.61%, compared with 6.95% for those with reported income. A chi-square test found this association to be statistically significant (χ² = 67.89, p < 0.001), suggesting that missingness itself contains predictive information. Therefore, the IncomeMissing feature was retained for modeling.

In [39]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(
    train_df['IncomeMissing'],
    train_df['SeriousDlqin2yrs']
)

chi2, p, dof, expected = chi2_contingency(contingency)

print(f"Chi-square statistic: {chi2:.2f}")
print(f"P-value: {p:.5f}")

Chi-square statistic: 67.89
P-value: 0.00000


# Results

In [40]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 13 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         150000 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

In [41]:
# Correlation of all numeric columns with the target
corr_with_target = (
    train_df.corr(numeric_only=True)["SeriousDlqin2yrs"]
    .sort_values(ascending=False)
)

print(corr_with_target)

SeriousDlqin2yrs                        1.000000
NumberOfTime30-59DaysPastDueNotWorse    0.125587
NumberOfTimes90DaysLate                 0.117175
NumberOfTime60-89DaysPastDueNotWorse    0.102261
NumberOfDependents                      0.046869
Unnamed: 0                              0.002801
RevolvingUtilizationOfUnsecuredLines   -0.001802
NumberRealEstateLoansOrLines           -0.007038
DebtRatio                              -0.007602
MonthlyIncome                          -0.017151
IncomeMissing                          -0.021308
NumberOfOpenCreditLinesAndLoans        -0.029669
age                                    -0.115386
Name: SeriousDlqin2yrs, dtype: float64


In [ ]:
train_df.to_csv("data/processed/train_clean.csv", index=False)